# Experiment 3 — PCA / activation-structure visualization

**Question.** Does the model's activation space contain visible structure that tracks what we
know about the data — does the same law cluster together across different surface forms
(the "semantic core"), and do complexity and representation type show up as ordered geometry?

**Design.**

1. **Dataset** — implication problems rendered **six ways each**, all deterministically (no LLM
   anywhere, so every tag is ground truth by construction): four themed stories (`graft`,
   `paint`, `signal`, `tea` — same law, near-disjoint vocabulary), one literal-NL description,
   and one Rigid-Grammar rendering. Renderers come from the
   [`semantic-drift-autoformalization`](https://github.com/shivam-raval96/semantic-drift-autoformalization)
   repo, pinned to the same commit as experiment 01.
2. **Activations** — one forward pass per text (no generation); residual-stream summaries
   (mean-pool primary, last-token alternative) at a sweep of layers, cached to Drive so the
   analysis cells re-run without a GPU.
3. **Analyses** — one section per tag, each with an eyeball plot and one number:
   **A** representation type (story / literal / RG), **B** complexity (total ops),
   **C** law identity across surface forms — the semantic-core test, quantified by 1-NN
   law retrieval, **D** lexical-confound baselines (embedding layer, TF-IDF).
4. **Reading the result** — exploratory by design: a null (undifferentiated blob) is fine and
   is reported as such. The interpretation cell at the bottom maps outcomes to decisions for
   experiments 1 and 7.

**Model.** `Qwen/Qwen3-4B` — same as experiment 01, so the geometry found here directly
informs the steering layer choice there.

**Runtime.** Colab GPU runtime (T4 is plenty — a few hundred short forward passes, no
generation). After the activation cache exists, everything below runs on CPU.

In [ ]:
# ------------------------------ Configuration ------------------------------

REPO_URL = "https://github.com/shivam-raval96/semantic-drift-autoformalization.git"
REPO_COMMIT = "c9e128f247b6daff5b47fd9711ded5e02d1095e9"  # same pin as experiment 01

MODEL_NAME = "Qwen/Qwen3-4B"

SEED = 0

# Dataset: PER_BIN pairs per total-operation bin 1..8 (stratified sampling),
# vacuous pairs dropped. Each surviving pair is rendered in six surface forms.
PER_BIN = 10

# Story themes rendered per pair. All four -> same law under near-disjoint
# vocabularies, which is what makes the law-identity test meaningful.
STORY_THEMES = ["graft", "paint", "signal", "tea"]

# Residual-stream read points, as hidden_states indices: hidden_states[L] is
# the output of decoder layer L-1 (1..36 for Qwen3-4B). Index 0 is the
# embedding stream and doubles as the in-model lexical baseline (section D).
LAYERS = [0, 6, 12, 18, 24, 30]

# Summary position for the headline plots: "pooled" (mean over tokens; robust
# to the very different text lengths) or "last". Both are always computed.
PRIMARY_POSITION = "pooled"

N_SHOW = 8       # pairs shown in the law-identity scatters (metrics use all pairs)
BATCH_SIZE = 8

# Smoke-test mode: tiny dataset, to validate the plumbing end to end in a
# couple of minutes before committing to a full run.
QUICK_TEST = False
if QUICK_TEST:
    PER_BIN, LAYERS = 2, [0, 12, 24]

In [ ]:
# ------------------------- Environment and outputs -------------------------
# Installs dependencies, clones the (public) repo that provides the dataset
# renderers, and picks an output directory. If Google Drive mounts, the
# activation cache is checkpointed there and survives a runtime reset;
# otherwise outputs stay in the local (ephemeral) filesystem.

%pip install -q -U "transformers>=4.51" accelerate scikit-learn matplotlib

import hashlib
import json
import subprocess
import sys
from pathlib import Path

if not Path("semantic-drift-autoformalization").exists():
    subprocess.run(["git", "clone", "-q", REPO_URL], check=True)
subprocess.run(
    ["git", "-C", "semantic-drift-autoformalization", "checkout", "-q", REPO_COMMIT],
    check=True,
)
sys.path.insert(0, str(Path("semantic-drift-autoformalization/informalizing-etp").resolve()))

from benchmark import drop_vacuous, load_equations, sample_pairs_stratified, synthesize_response
from literalform import render_description
from storyform import render_story

import numpy as np
import torch
from tqdm.auto import tqdm

OUT_DIR = Path("exp03-outputs")
try:
    from google.colab import drive

    drive.mount("/content/drive")
    OUT_DIR = Path("/content/drive/MyDrive/mech-interp-experiments/exp03-pca")
except Exception as error:
    print(f"Google Drive not available ({error}); using local {OUT_DIR}/")
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("outputs ->", OUT_DIR.resolve())

## Dataset — six deterministic surface forms per problem

Each problem is "if a world always obeys rule E, must it also obey rule F?", sampled with the
seeded, complexity-stratified sampler from `benchmark.py` (vacuous laws dropped, following the
repo's convention). Every pair is then rendered six ways by deterministic code:

- **4 themed stories** — `render_story` with an explicitly pinned theme. The four themes use
  different settings, agents, and palette vocabularies, so two stories about the same law share
  almost no content words. If same-law stories still land together in activation space, that
  cannot be surface overlap.
- **1 literal-NL description** — `render_description`: dry English with variables and named
  intermediate values, no narrative.
- **1 Rigid-Grammar rendering** — `synthesize_response` on the pair's own canonical metadata:
  the two-line `ASSUME: … / ASK: …` form, exactly what the formalization task asks models to
  produce.

Every text carries ground-truth tags — `pair_id` (law identity), form, theme, total operation
count, nesting depth — trustworthy by construction since no LLM touches the pipeline. This is
the "the data must contain genuine structure" requirement: the tags *are* the structure we then
look for in activation space.

In [ ]:
# ------------------------------ Build the dataset ------------------------------

equations, equations_sha = load_equations()
print(f"ETP equation list: {len(equations)} equations (sha256 {equations_sha[:12]})")

pairs = drop_vacuous(sample_pairs_stratified(equations, PER_BIN, SEED, form="story"))
print(f"{len(pairs)} pairs after dropping vacuous laws")

records, dropped = [], []
for sample in pairs:
    meta = sample["metadata"]
    tags = {
        "pair_id": sample["pair_id"],
        "ops_total": sample["ops_total"],
        "depth": sample["depth"],
    }
    try:
        rendered = [
            {**tags, "form": "story", "theme": theme,
             "text": render_story(meta["equation_e"], meta["equation_f"], theme_key=theme)[0]}
            for theme in STORY_THEMES
        ]
        rendered.append({**tags, "form": "literal", "theme": "literal",
                         "text": render_description(meta["equation_e"], meta["equation_f"])[0]})
        rendered.append({**tags, "form": "rg", "theme": "rg",
                         "text": synthesize_response(meta)})
    except (ValueError, KeyError) as error:
        dropped.append((sample["pair_id"], str(error)))  # keep the 6-form grid complete
        continue
    records.extend(rendered)

texts = [r["text"] for r in records]
n_pairs = len({r["pair_id"] for r in records})
print(f"{n_pairs} pairs x 6 forms = {len(records)} texts"
      + (f"  ({len(dropped)} pairs dropped: {dropped})" if dropped else ""))

example_id = records[0]["pair_id"]
print(f"\n--- pair {example_id} in three of its six forms ---")
for record in records:
    if record["pair_id"] == example_id and record["theme"] in ("paint", "literal", "rg"):
        print(f"\n[{record['theme']}]\n{record['text']}")

## Activation extraction

One forward pass per **bare text** — deliberately not the formalization prompts, for the same
reason as experiment 01: the arms use different prompt templates, and prompted inputs would
bake template wording into the geometry rather than the story-vs-abstract difference.

At each probed layer we keep two summaries per text: the **mean-pooled** vector (primary here —
the six forms differ hugely in length, and the last token of a story is narratively arbitrary)
and the **last-token** vector (the robustness alternative; also what experiment 01's steering
vector uses).

The model is only loaded if the cache is missing, so on a cached rerun this notebook never
touches the GPU. Storage is small (~n_texts × layers × 2 × hidden ≈ tens of MB).

In [ ]:
# ---------------------- Extract (or reload) the activations ----------------------

DATASET_SHA = hashlib.sha256(json.dumps([MODEL_NAME, LAYERS, texts]).encode()).hexdigest()[:12]
CACHE = OUT_DIR / f"activations-{DATASET_SHA}.pt"
RECORDS_PATH = OUT_DIR / f"records-{DATASET_SHA}.jsonl"


@torch.no_grad()
def collect_residuals(texts, layers):
    """Residual-stream summaries per text: last-token and mean-pooled vectors
    at each requested hidden_states index."""
    tokenizer.padding_side = "right"  # keeps last-token indexing by length valid
    last = {L: [] for L in layers}
    pooled = {L: [] for L in layers}
    for i in tqdm(range(0, len(texts), BATCH_SIZE), desc="activations"):
        encoded = tokenizer(texts[i:i + BATCH_SIZE], return_tensors="pt", padding=True)
        encoded = encoded.to(model.device)
        output = model(**encoded, output_hidden_states=True)
        mask = encoded["attention_mask"]
        lengths = mask.sum(dim=1)
        row_idx = torch.arange(mask.size(0), device=mask.device)
        for L in layers:
            hidden = output.hidden_states[L].float()
            last[L].append(hidden[row_idx, lengths - 1].cpu())
            mean_tok = (hidden * mask.unsqueeze(-1)).sum(dim=1) / lengths.unsqueeze(-1)
            pooled[L].append(mean_tok.cpu())
        del output
    concat = lambda parts: {L: torch.cat(v) for L, v in parts.items()}
    return {"last": concat(last), "pooled": concat(pooled)}


if CACHE.exists():
    acts = torch.load(CACHE)
    print(f"loaded cached activations from {CACHE.name}")
else:
    from transformers import AutoModelForCausalLM, AutoTokenizer

    if torch.cuda.is_available():
        dtype = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16
    else:
        dtype = torch.float32
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=dtype, device_map="auto")
    model.eval()
    assert max(LAYERS) <= model.config.num_hidden_layers

    acts = collect_residuals(texts, LAYERS)
    torch.save(acts, CACHE)
    RECORDS_PATH.write_text("\n".join(json.dumps(r) for r in records) + "\n")
    print(f"saved activations -> {CACHE.name}")

hidden_dim = acts["pooled"][LAYERS[0]].shape[1]
print(f"{len(records)} texts x {len(LAYERS)} layers x 2 positions, hidden dim {hidden_dim}")

In [ ]:
# ------------------------------ Analysis helpers ------------------------------
# Everything below is CPU-only and reruns instantly from the cache.

import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

forms = np.array([r["form"] for r in records])       # story / literal / rg
themes = np.array([r["theme"] for r in records])     # 4 themes + literal + rg
surfaces = themes                                    # the six surface groups
pair_ids = np.array([r["pair_id"] for r in records])
ops = np.array([r["ops_total"] for r in records])

SURFACE_STYLE = {  # color, marker
    "graft":   ("#8c564b", "o"),
    "paint":   ("#e377c2", "o"),
    "signal":  ("#1f77b4", "o"),
    "tea":     ("#ff7f0e", "o"),
    "literal": ("#2ca02c", "^"),
    "rg":      ("#000000", "s"),
}
FORM_MARKER = {"story": "o", "literal": "^", "rg": "s"}


def matrix(position, layer):
    return acts[position][layer].float().numpy()


def pca2(X):
    """2D PCA projection and the variance it explains."""
    p = PCA(n_components=2).fit(X)
    return p.transform(X), float(p.explained_variance_ratio_.sum())


def layer_grid(title, rows=1):
    fig, axes = plt.subplots(
        rows, len(LAYERS), figsize=(3.3 * len(LAYERS), 3.5 * rows), squeeze=False)
    fig.suptitle(title)
    for ax in axes.flat:
        ax.set_xticks([])
        ax.set_yticks([])
    return fig, axes


def cosine_sims(X):
    Xn = X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-9)
    return Xn @ Xn.T


def law_retrieval_all(X):
    """1-NN law retrieval, all surfaces: the nearest neighbour among texts of a
    *different* surface (another theme, or another form) must encode the same pair."""
    sims = cosine_sims(X)
    sims[surfaces[:, None] == surfaces[None, :]] = -np.inf  # also removes self
    nn = sims.argmax(axis=1)
    return float((pair_ids[nn] == pair_ids).mean())


def law_retrieval_story(X):
    """Story-only variant: nearest neighbour among stories of *other themes* —
    near-disjoint vocabulary by construction, the purest semantic-core test."""
    sims = cosine_sims(X)
    story = forms == "story"
    allowed = story[None, :] & (themes[:, None] != themes[None, :])
    sims[~allowed] = -np.inf
    nn = sims.argmax(axis=1)
    return float((pair_ids[nn] == pair_ids)[story].mean())

## A — Representation type

Global PCA per layer, one point per text, styled by surface (circles = stories, one color per
theme; green triangles = literal; black squares = RG). The dashed path connects the story,
literal, and RG centroids.

What to look for: form should separate crisply (if even this fails, deeper structure is
unlikely at these read points). The interesting question is *how* it separates — if the literal
centroid sits **between** story and RG, that is the first hint of experiment 7's ordered
formality ladder.

In [ ]:
# ---------------------- A: PCA colored by surface form ----------------------

fig, axes = layer_grid(f"PCA by surface form  ({PRIMARY_POSITION} position)")
for ax, L in zip(axes[0], LAYERS):
    X2, evr = pca2(matrix(PRIMARY_POSITION, L))
    for key, (color, marker) in SURFACE_STYLE.items():
        m = surfaces == key
        ax.scatter(X2[m, 0], X2[m, 1], s=12, c=color, marker=marker, alpha=0.7, label=key)
    centroids = [X2[forms == f].mean(axis=0) for f in ("story", "literal", "rg")]
    ax.plot(*zip(*centroids), "k--", lw=1, zorder=3)
    for (x, y), label in zip(centroids, ("S", "L", "RG")):
        ax.annotate(label, (x, y), fontsize=10, fontweight="bold", zorder=4)
    ax.set_title(f"hidden_states[{L}]  ({evr:.0%} var in 2D)", fontsize=9)
axes[0][0].legend(fontsize=7, loc="best")
plt.tight_layout()
plt.show()

## B — Complexity

Same projections, colored by the pair's total operation count (1–8). Top row: all forms
(complexity may be invisible while the form signal dominates the top PCs). Bottom row: PCA
refit on stories only, where a complexity gradient has more room to surface.

What to look for: an ordered gradient or fan, not clusters — complexity is ordinal, so the
interesting outcome is *geometry that respects the order*.

In [ ]:
# ---------------------- B: PCA colored by complexity ----------------------

story_mask = forms == "story"

fig, axes = layer_grid("PCA by total operation count", rows=2)
for col, L in enumerate(LAYERS):
    X2, evr = pca2(matrix(PRIMARY_POSITION, L))
    sc = axes[0][col].scatter(X2[:, 0], X2[:, 1], c=ops, s=12, cmap="viridis", alpha=0.8)
    axes[0][col].set_title(f"hidden_states[{L}]  ({evr:.0%} var)", fontsize=9)

    Xs2, evr_s = pca2(matrix(PRIMARY_POSITION, L)[story_mask])
    axes[1][col].scatter(Xs2[:, 0], Xs2[:, 1], c=ops[story_mask], s=12, cmap="viridis", alpha=0.8)
    axes[1][col].set_title(f"stories only  ({evr_s:.0%} var)", fontsize=9)

axes[0][0].set_ylabel("all forms")
axes[1][0].set_ylabel("stories only")
fig.colorbar(sc, ax=axes.ravel().tolist(), shrink=0.7, label="total ops")
plt.show()

## C — Law identity across surface forms (the semantic-core test)

The headline analysis. If activations of differently-themed stories cluster by the law they
encode, that is early evidence for a representation-invariant semantic core — the premise
behind experiment 7's H1.

**Metric first, pictures second.** The quantitative test is **1-NN law retrieval** in the full
high-dimensional space (PCA's top 2 components can hide real structure, so the metric must not
depend on the projection):

- *all surfaces* — for each text, is its cosine nearest neighbour **among texts of a different
  surface** (other theme or other form) the same pair? Chance ≈ 1/n_pairs.
- *story × theme* — story queries, candidates restricted to stories of **other themes**. Themes
  share almost no vocabulary, so this variant is immune to lexical overlap by construction.

Then two scatters at the best-retrieving layer:

1. **stories only** — PCA over story activations, `N_SHOW` pairs colored (rest gray). Same-law
   dots of four different themes landing together = the semantic core, visible.
2. **all six forms, surface-mean-subtracted** — each surface group's centroid is removed
   (deleting the dominant form/register signal of section A), then PCA. Colors = pairs, markers
   = form. This is the cross-form version: story, literal, *and* RG renderings of one law
   co-clustering.

In [ ]:
# ------------------- C: 1-NN law retrieval, then the scatters -------------------

retrieval = {
    (pos, L): (law_retrieval_all(matrix(pos, L)), law_retrieval_story(matrix(pos, L)))
    for pos in ("pooled", "last")
    for L in LAYERS
}

print(f"1-NN law retrieval accuracy  (chance ~ 1/{n_pairs} = {1 / n_pairs:.1%})")
print(f"{'layer':>5}   {'pooled all':>10} {'story x-theme':>14}   {'last all':>10} {'story x-theme':>14}")
for L in LAYERS:
    pa, ps = retrieval[("pooled", L)]
    la, ls = retrieval[("last", L)]
    note = "  <- embeddings (lexical baseline)" if L == 0 else ""
    print(f"{L:>5}   {pa:>10.1%} {ps:>14.1%}   {la:>10.1%} {ls:>14.1%}{note}")

best_layer = max((L for L in LAYERS if L > 0),
                 key=lambda L: retrieval[(PRIMARY_POSITION, L)][0])
print(f"\nbest layer by {PRIMARY_POSITION} all-surface retrieval: hidden_states[{best_layer}]")

# ----- scatters at the best layer -----

# Pairs to highlight, spread across the complexity range.
unique_pairs = sorted({(p, o) for p, o in zip(pair_ids, ops)}, key=lambda t: (t[1], t[0]))
show_idx = np.linspace(0, len(unique_pairs) - 1, min(N_SHOW, len(unique_pairs))).astype(int)
shown = [unique_pairs[i][0] for i in show_idx]
pair_color = {p: plt.get_cmap("tab10")(i % 10) for i, p in enumerate(shown)}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5.5))

# (1) stories only, colored by pair
Xs2, evr_s = pca2(matrix(PRIMARY_POSITION, best_layer)[story_mask])
story_pairs = pair_ids[story_mask]
background = ~np.isin(story_pairs, shown)
ax1.scatter(Xs2[background, 0], Xs2[background, 1], s=10, c="lightgray", alpha=0.5)
for p in shown:
    m = story_pairs == p
    ax1.scatter(Xs2[m, 0], Xs2[m, 1], s=40, color=pair_color[p], label=p)
ax1.set_title(f"stories only, colored by law  (layer {best_layer}, {evr_s:.0%} var)")
ax1.legend(fontsize=7, title="pair", loc="best")

# (2) all six forms, surface-mean-subtracted
X = matrix(PRIMARY_POSITION, best_layer).copy()
for s in np.unique(surfaces):
    X[surfaces == s] -= X[surfaces == s].mean(axis=0)
Xc2, evr_c = pca2(X)
background = ~np.isin(pair_ids, shown)
ax2.scatter(Xc2[background, 0], Xc2[background, 1], s=10, c="lightgray", alpha=0.4)
for p in shown:
    for form, marker in FORM_MARKER.items():
        m = (pair_ids == p) & (forms == form)
        ax2.scatter(Xc2[m, 0], Xc2[m, 1], s=40, color=pair_color[p], marker=marker)
ax2.set_title(f"all forms, surface-mean-subtracted  (layer {best_layer}, {evr_c:.0%} var)")
for ax in (ax1, ax2):
    ax.set_xticks([])
    ax.set_yticks([])
plt.tight_layout()
plt.show()

## D — Lexical-confound baselines

The most likely *false positive* here is clustering driven by surface features rather than
meaning. Two baselines guard against that:

- **layer 0** (embedding stream, already on the retrieval curves) — what the model "sees"
  before any computation: purely lexical;
- **TF-IDF bag-of-words** of the raw texts — model-free surface overlap.

Reading: if mid-stack retrieval clearly beats both baselines, the law clustering is computed
semantics, not shared wording. If the baselines already match the best layer, the "structure"
is trivial and section C's result should be discounted — especially for the *all surfaces*
metric, where literal and RG renderings of one law do share variable names and operator
wording. The *story × theme* metric should hold near baseline-zero regardless, because themes
were built with disjoint vocabularies.

In [ ]:
# ------------------- D: baselines and the summary figure -------------------

from sklearn.feature_extraction.text import TfidfVectorizer

X_tfidf = np.asarray(TfidfVectorizer().fit_transform(texts).todense())
tfidf_all = law_retrieval_all(X_tfidf)
tfidf_story = law_retrieval_story(X_tfidf)
print(f"TF-IDF baseline: all surfaces {tfidf_all:.1%}, story x-theme {tfidf_story:.1%}")

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(LAYERS, [retrieval[("pooled", L)][0] for L in LAYERS], "-o",
        label="pooled, all surfaces")
ax.plot(LAYERS, [retrieval[("pooled", L)][1] for L in LAYERS], "-^",
        label="pooled, story x-theme")
ax.plot(LAYERS, [retrieval[("last", L)][0] for L in LAYERS], "--s",
        alpha=0.6, label="last token, all surfaces")
ax.axhline(tfidf_all, color="red", linestyle="--", lw=1,
           label=f"TF-IDF all {tfidf_all:.0%}")
ax.axhline(tfidf_story, color="darkred", linestyle=":", lw=1,
           label=f"TF-IDF story x-theme {tfidf_story:.0%}")
ax.axhline(1 / n_pairs, color="gray", linestyle=":", lw=1,
           label=f"chance {1 / n_pairs:.1%}")
ax.annotate("embeddings\n(lexical)", (LAYERS[0], retrieval[("pooled", LAYERS[0])][0]),
            fontsize=8, textcoords="offset points", xytext=(6, 6))
ax.set_xlabel("hidden_states index")
ax.set_ylabel("1-NN law retrieval accuracy")
ax.set_ylim(0, 1.02)
ax.set_title("Where in the stack does the semantic core emerge?")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## Interpretation notes

Fill in after the run:

- **A (form).** Did surfaces separate? Does the literal centroid sit between story and RG
  (first hint of the formality ladder)?
- **B (complexity).** Gradient, fan, or nothing? All-forms vs stories-only?
- **C (law identity).** Best layer and retrieval numbers vs chance, for both metrics. Do the
  scatters show the same thing the metric does?
- **D (baselines).** Margin of the best layer over layer 0 and TF-IDF. Does the story × theme
  metric survive the disjoint-vocabulary test?

**Decision mapping:**

- *Law clustering across themes/forms well above the lexical baselines* → green light for
  experiment 7's H1 (a law-identity component invariant across the ladder is plausible). The
  best layer here is the first candidate read/steer point for experiments 1 and 7.
- *Form separates but law does not* → register is strongly represented at these read points,
  the semantic core is not visible → H1's factorization is at risk; experiment 1 (steering the
  form direction) is unaffected.
- *Blob everywhere* → check the last-token alternative (`PRIMARY_POSITION = "last"`) before
  concluding; a PCA/1-NN null at two positions is still weak evidence of absence — report as
  "no visible structure at these read points" and move on, per the timebox.

Follow-ups if positive: color by ASSUME-law only (`label_e`) instead of the full pair — do
different questions about the same law co-cluster?; per-complexity-bin retrieval (does the
semantic core degrade with nesting depth?); and the fitted formality curve + probes, which
belong to experiment 7, not here.